In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%cd /home/albin/egna_proj/block_puzzle_rl/

/home/albin/egna_proj/block_puzzle_rl


/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# your modules
from agent.dqn_agent import DQNAgent
from agent.utils import encode_state
from game.block_puzzle_env import BlockPuzzleEnv
from gymnasium.wrappers import TimeLimit

In [4]:
torch.cuda.is_available()

True

In [5]:
raw_env = BlockPuzzleEnv(width=12, height=10, num_blocks=3)
env = TimeLimit(raw_env, max_episode_steps=1000)
env.env.render(mode='human')

□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □


In [6]:
obs_dict = env.reset()
flat_obs = encode_state(obs_dict)
state_dim = flat_obs.shape[0]
action_dim = int(np.prod(env.action_space.nvec))

In [7]:
agent = DQNAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    lr=1e-3,
    gamma=0.99,
    epsilon=1.0,
    epsilon_decay=0.9995,
    epsilon_min=0.01,
)

NUM_EPISODES = 2000
TARGET_UPDATE_FREQ = 1000

# Storage
all_rewards = []
all_lengths = []
global_step = 0


In [ ]:
# Training loop


for episode in range(NUM_EPISODES):
    obs_dict = env.reset()
    state = encode_state(obs_dict)
    ep_reward = 0
    ep_length = 0
    done = False
    
    while not done:
        flat_a = agent.select_action(state)
        
        a0, a1, a2 = np.unravel_index(flat_a, env.action_space.nvec)
        action = (int(a0), int(a1), int(a2))
        
        (next_obs_dict, reward, terminated, truncated, info) = env.step(action)
        done = terminated or truncated
        next_state = encode_state(next_obs_dict)
        
        agent.replay_buffer.push(
            state=state,
            action=flat_a,
            reward=reward,
            next_state=next_state,
            done=done
        )
        
        agent.train_step()
        
        state = next_state
        ep_reward += reward
        ep_length += 1
        global_step += 1
        
        if global_step % TARGET_UPDATE_FREQ == 0:
            agent.update_target()
        
        agent.decay_epsilon()
    
    all_rewards.append(ep_reward)
    all_lengths.append(ep_length)
    if episode % 1 == 0:
        print(f"Episode {episode + 1}/{NUM_EPISODES} | "
              f"Reward: {ep_reward:.2f} | "
              f"Length: {ep_length} | "
              f"Epsilon: {agent.epsilon:.2f}")
    

Episode 1/2000 | Reward: -95.00 | Length: 2 | Epsilon: 1.00
Episode 2/2000 | Reward: -85.00 | Length: 4 | Epsilon: 1.00
Episode 3/2000 | Reward: -95.00 | Length: 2 | Epsilon: 1.00
Episode 4/2000 | Reward: -95.00 | Length: 2 | Epsilon: 1.00
Episode 5/2000 | Reward: -100.00 | Length: 1 | Epsilon: 0.99
Episode 6/2000 | Reward: -100.00 | Length: 1 | Epsilon: 0.99
Episode 7/2000 | Reward: -90.00 | Length: 3 | Epsilon: 0.99
Episode 8/2000 | Reward: -90.00 | Length: 3 | Epsilon: 0.99
Episode 9/2000 | Reward: -90.00 | Length: 3 | Epsilon: 0.99
Episode 10/2000 | Reward: -90.00 | Length: 3 | Epsilon: 0.99
Episode 11/2000 | Reward: -95.00 | Length: 2 | Epsilon: 0.99
Episode 12/2000 | Reward: -95.00 | Length: 2 | Epsilon: 0.99
Episode 13/2000 | Reward: -95.00 | Length: 2 | Epsilon: 0.99
Episode 14/2000 | Reward: -95.00 | Length: 2 | Epsilon: 0.98
Episode 15/2000 | Reward: -95.00 | Length: 2 | Epsilon: 0.98
Episode 16/2000 | Reward: -100.00 | Length: 1 | Epsilon: 0.98
Episode 17/2000 | Reward: -90.

In [ ]:
# %% evaluation cell
import time
import numpy as np

# Unwrap the TimeLimit wrapper to call your env’s render()
# Replace `env` here with your actual env variable if different
raw_env = env.env if hasattr(env, 'env') else env

# Ensure render prints to stdout
raw_env.render_mode = 'human'

# Turn off exploration: purely greedy
agent.epsilon = 0.0

# Reset and render initial state
obs_dict = raw_env.reset()
state = encode_state(obs_dict)
print("Initial grid:")
raw_env.render()

done = False
step = 0

while not done and step < 200:
    # Greedy action
    flat_a = agent.select_action(state)
    a0, a1, a2 = np.unravel_index(flat_a, raw_env.action_space.nvec)
    action = (int(a0), int(a1), int(a2))
    
    # Step and get reward
    next_obs, reward, terminated, truncated, _ = raw_env.step(action)
    state = encode_state(next_obs)
    
    print(f"\nStep {step+1}: place block {a0} at row {a1}, col {a2} → reward {reward:.1f}")
    raw_env.render()
    
    done = terminated or truncated
    step += 1
    time.sleep(0.3)  # slow down so you can observe

if done:
    print(f"\nEpisode finished after {step} steps with total score {raw_env.game.score}")
else:
    print(f"\nStopped after {step} steps (time limit or step cap)")


Initial grid:
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

Step 1: place block 1 at row 1, col 2 → reward 5.0
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ ■ □ □ □ □ □ □ □
□ □ ■ ■ ■ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

Step 2: place block 2 at row 0, col 0 → reward 5.0
■ ■ ■ □ □ □ □ □ □ □ □ □
■ ■ ■ □ ■ □ □ □ □ □ □ □
□ □ ■ ■ ■ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

Step 3: place block 1 at row 1, col 2 → reward -100.0
■ ■ ■ □ □ □ □ □ □ □ □ □
■ ■ ■ □ ■ □ □ □ □ □ □ □
□ □ ■ ■ ■ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □